# Lab10 part 2: Conversational AI with Retrieval-Augmented Generation (RAG)

## Introduction

In this part of the lab, we extend the Retrieval-Augmented Generation (RAG) framework to build a **Conversational AI system**. Unlike the previous section, where each query was treated independently, here the system will maintain **conversation memory** to provide context-aware responses.

### How Conversational RAG Works
1. **Document Retrieval**: As before, the system retrieves relevant documents for the current query.
2. **Conversation Memory**: The system tracks the conversation's history, allowing responses to build on prior queries and answers.
3. **LLM Integration**: The retrieved documents and conversation history are used as input to the LLM for generating responses.

Maintaining conversation memory can improve the user experience by enabling continuity, coherence, and context-awareness.

---

## Exercise 1: Building Conversational RAG with Memory

### Objective
Extend the RAG pipeline to maintain a memory of previous queries and responses while answering user queries.

### Instructions
1. Modify the RAG pipeline to include a **memory mechanism** that:
   - Stores the last `N` queries and responses (e.g., `N=3`).
   - Appends this memory to the input prompt for the LLM.
2. Update the prompt to include:
   - The current query.
   - Relevant retrieved context.
   - Conversation memory (e.g., a list of recent exchanges).
3. Use the same models as before:
   - Document Encoder: Sentence transformer.
   - Generators: Llama 3.2 (3B), Gemma 3 (4B).
4. Experiment with memory size (`N`) and prompt formatting to balance coherence and performance.

### Tasks
- Test the conversational RAG by asking a series of related queries.
- Analyze how the responses differ when memory size (`N`) changes.
- Experiment with prompt design to ensure the system sticks to retrieved context and conversation history.


## Exercise 2: Selective Retrieval from Conversational History

### Objective
Instead of appending the entire conversation history to the prompt, implement a system that saves the full history in a file and retrieves only the relevant exchanges for a given query. This ensures that the context is concise and relevant, improving efficiency and coherence.

---

### Instructions

1. **Save and Maintain Conversation History**:  
   - Save the complete conversation history (queries and responses) into a structured file (e.g., JSON or CSV).
   - Ensure the file format includes:
     - **User Query**: The input from the user.
     - **LLM Response**: The generated response.
     - **Timestamps**: To maintain chronological order and for debugging purposes.

2. **Combine Retrieval Sources**:  
   - When a new query is received:
     - Retrieve **immediate previous exchanges** to maintain conversational flow.
     - Use a **retriever model** (e.g., Sentence Transformer) to identify the top `k` most similar exchanges from the saved history file.
     - Retrieve relevant sentences from the **knowledge base** using the same retriever.

3. **Prompt Construction**:  
   - Construct a concise and informative prompt for the LLM by including:
     - Relevant parts of the **previous conversation**.
     - The **top `k` exchanges** from the saved history file.
     - Relevant sentences from the **knowledge base**.
     - The **current query**.

4. **System Testing**:  
   - Use the same models as in previous exercises:
     - Document Encoder: Sentence Transformer.
     - Generators: Llama 3.2 (3B), Gemma 3 (4B).
   - Experiment with retrieval parameters such as:
     - `k` (number of most similar exchanges to retrieve from history).
     - Scoring threshold for relevance in the retriever model.

---

## Exercise 3 (extra): Adding Multi-Turn PDF Knowledge Base

### Objective
Integrate the Conversational RAG system with a PDF knowledge base, enabling it to handle multi-turn conversations grounded in external documents.

### Instructions
1. Prepare a folder (`input_folder`) with PDFs as in the previous lab.
2. Extend the preprocessing pipeline to:
   - Extract text, tokenize sentences, and create embeddings for each PDF.
   - Store embeddings in a format optimized for retrieval during multi-turn conversations.
3. Update the inference pipeline to:
   - Match each query with the PDF embeddings.
   - Incorporate conversation memory into the context for the LLM.
4. Test the system by asking a series of related queries about the PDF content.

### Tasks
- Test the multi-turn Conversational RAG system with various PDF files.
- Analyze how memory affects the system's ability to provide coherent, context-aware responses.



In [1]:
# ============================================================================
# SECTION 1: INSTALLATION
# ============================================================================

# Install PyTorch with GPU support (CUDA 12.1 for modern GPUs like T4, A100, V100)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install core libraries with specific versions for reproducibility
!pip install transformers         # Hugging Face transformers for LLMs
!pip install sentence-transformers  # For encoding text into embeddings
!pip install nltk                   # Natural Language Toolkit for BLEU scores
!pip install numpy               # Numerical operations
!pip install accelerate           # Efficient model loading and memory management
!pip install pypdf

# Download required NLTK data for tokenization and BLEU score calculation
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 9.7 MB/s eta 0:00:00


True

In [2]:
# ============================================================================
# SECTION 2: EXERCISE 1 - CONVERSATIONAL RAG WITH MEMORY
# ============================================================================

import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load Models ---
print("Loading models...")
retriever = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
generator = AutoModelForCausalLM.from_pretrained("unsloth/Llama-3.2-3B-Instruct", torch_dtype=torch.float16, device_map="auto")

# --- Knowledge Base ---
knowledge_base = [
    "The capital of France is Paris.",
    "The Eiffel Tower was constructed in 1889.",
    "Python is a high-level programming language.",
    "RAG stands for Retrieval-Augmented Generation.",
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with multiple layers.",
    "Natural Language Processing (NLP) focuses on text and language understanding.",
    "Transformers are the architecture behind modern LLMs."
]
kb_embeddings = retriever.encode(knowledge_base, convert_to_tensor=True)

MEMORY_SIZE = 3
chat_history = []  # List of tuples: (user_query, ai_response)

# --- 1. MEMORY FORMATTING ---

def format_history(history):
    """
    Goal: Prepare the conversation history so the LLM remembers previous turns.
    Input: history (list of tuples) -> The global chat_history list.
    Output: formatted (string) -> A single string representing the recent dialogue.
    """
    # TODO: 1.1. Initialize an empty string variable named 'formatted'.
    formatted = ""

    # TODO: 1.2. Slice the 'history' list to only include the last 'MEMORY_SIZE' items.
    # Instruction: This prevents the prompt from growing too long and exceeding the model's token limit.
    recent_history = history[-MEMORY_SIZE:]

    # TODO: 1.3. Iterate through the sliced history.
    # Instruction: For each (user_text, ai_text) pair, append them to 'formatted'
    # in this style: "User: {user_text}\nAssistant: {ai_text}\n"
    for user_text, ai_text in recent_history:
        formatted += f"User: {user_text}\nAssistant: {ai_text}\n"

    # TODO: 1.4. Return the 'formatted' string.
    return formatted


# --- 2. RAG GENERATION LOGIC ---

def chat_generate(query):
    """
    Goal: Implement the "Retrieve-Augment-Generate" flow with conversational memory.
    Input: query (string) -> The user's new question.
    Output: response (string) -> The final answer generated by the model.
    """

    # --- SUB-STEP A: RETRIEVAL ---
    # Goal: Find the most relevant facts from the Knowledge Base (KB).

    # TODO: 2.1. Encode the 'query' string into a vector using 'retriever.encode'.
    # Note: Use convert_to_tensor=True to keep it on the GPU/CPU.
    query_emb = retriever.encode(query, convert_to_tensor=True)

    # TODO: 2.2. Compute similarity between your query vector and 'kb_embeddings'.
    # Hint: Use 'util.cos_sim' from the sentence_transformers library.
    scores = util.cos_sim(query_emb, kb_embeddings)[0]

    # TODO: 2.3. Extract the top 2 indices with the highest similarity scores.
    # Hint: Use 'torch.topk(scores, k=2).indices'.
    top_indices = torch.topk(scores, k=2).indices

    # TODO: 2.4. Map those indices back to the 'knowledge_base' text list.
    # Instruction: Join the results into a single string called 'context' separated by newlines.
    context = "\n".join([knowledge_base[idx] for idx in top_indices])


    # --- SUB-STEP B: PROMPT ENGINEERING ---
    # Goal: Combine System Rules, Context, History, and Query into a single Llama-3 prompt.

    # TODO: 2.5. Get the formatted history string by calling 'format_history(chat_history)'.
    history_text = format_history(chat_history)

    # TODO: 2.6. Create a 'system_prompt' string.
    # Instruction: Tell the AI it is a technical expert. Command it to ONLY use
    # the provided context. Tell it to say "I don't know" if the answer isn't there.
    system_prompt = """You are a technical expert assistant. You must ONLY use the provided context to answer questions. If the context does not contain the answer, respond with 'I don't know'."""

    # TODO: 2.7. Construct 'full_prompt' using Llama-3's specific Chat Template.
    # Detailed Formatting Instructions:
    # 1. Start with "<|begin_of_text|>"
    # 2. Add System Header: "<|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|>"
    # 3. Add User Header: "<|start_header_id|>user<|end_header_id|>\n\n"
    #    Inside this block, clearly label:
    #    "Context: {context}\n"
    #    "Chat History: {history_text}\n"
    #    "New Question: {query}<|eot_id|>"
    # 4. Add Assistant Header: "<|start_header_id|>assistant<|end_header_id|>\n\n"
    full_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

Context: {context}
Chat History: {history_text}
New Question: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""


    # --- SUB-STEP C: GENERATION ---
    # Goal: Run the model and clean the output.

    # TODO: 2.8. Tokenize 'full_prompt' and move the resulting 'inputs' to the 'device'.
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    # TODO: 2.9. Generate tokens using 'generator.generate'.
    # Instruction: Set max_new_tokens=150 and do_sample=False (greedy decoding).
    with torch.no_grad():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # TODO: 2.10. Decode the output tokens.
    # CRITICAL: Slice the decoded text so you ONLY return the new text generated
    # AFTER the prompt (to avoid repeating the user's question back to them).
    response = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


    # --- SUB-STEP D: MEMORY UPDATE ---
    # TODO: 2.11. Append the (query, response) as a tuple to the global 'chat_history'.
    chat_history.append((query, response))

    # TODO: 2.12. Return the 'response' string.
    return response


# --- 3. THE INTERACTIVE LOOP ---

print("\nConversational RAG System Ready!")
print("Type 'exit' to quit.\n")

# TODO: 3.1. Create a loop that runs until the user types 'exit'.
while True:
    # TODO: 3.2. Inside the loop, take user input using 'input()'.
    user_input = input("You: ").strip()

    if user_input.lower() == 'exit':
        print("Goodbye!")
        break

    # TODO: 3.3. Call 'chat_generate(user_input)'.
    ai_response = chat_generate(user_input)

    # TODO: 3.4. Print the AI's response to the console.
    print(f"AI: {ai_response}\n")

Loading models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (17 outputs left out)

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]


Conversational RAG System Ready!
Type 'exit' to quit.

You: hello


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


AI: I don't know

You: what is RAG
AI: RAG stands for Retrieval-Augmented Generation.

You: what is the capital of france
AI: Paris

You: exit
Goodbye!


In [3]:
# ============================================================================
# SECTION 3: EXERCISE 2 - SELECTIVE RETRIEVAL FROM CONVERSATIONAL HISTORY
# ============================================================================

import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load Models ---
print("Loading models...")
retriever = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
generator = AutoModelForCausalLM.from_pretrained("unsloth/Llama-3.2-3B-Instruct", torch_dtype=torch.float16, device_map="auto")

# --- Knowledge Base ---
knowledge_base = [
    "The capital of France is Paris.",
    "The Eiffel Tower was constructed in 1889.",
    "Python is a high-level programming language.",
    "RAG stands for Retrieval-Augmented Generation.",
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with multiple layers.",
    "Natural Language Processing (NLP) focuses on text and language understanding.",
    "Transformers are the architecture behind modern LLMs."
]
kb_embeddings = retriever.encode(knowledge_base, convert_to_tensor=True)

HISTORY_FILE = "conversation_history.json"
IMMEDIATE_MEMORY = 2  # Number of immediate previous exchanges to include
TOP_K_HISTORY = 2     # Number of most similar past exchanges to retrieve

# --- Helper Functions ---

def save_to_history(query, response):
    """Save a conversation turn to the history file."""
    try:
        with open(HISTORY_FILE, 'r') as f:
            history = json.load(f)
    except FileNotFoundError:
        history = []

    history.append({
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "response": response
    })

    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)

def load_history():
    """Load the complete conversation history from file."""
    try:
        with open(HISTORY_FILE, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return []

def retrieve_relevant_history(query, k=TOP_K_HISTORY):
    """Retrieve the top-k most similar exchanges from history."""
    history = load_history()
    if not history:
        return []

    # Create embeddings for all past queries
    past_queries = [h["query"] for h in history]
    past_embeddings = retriever.encode(past_queries, convert_to_tensor=True)

    # Find most similar
    query_emb = retriever.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, past_embeddings)[0]
    top_indices = torch.topk(scores, k=min(k, len(history))).indices

    return [history[idx] for idx in top_indices]

def get_immediate_history(n=IMMEDIATE_MEMORY):
    """Get the last n exchanges from history."""
    history = load_history()
    return history[-n:] if len(history) > 0 else []

def selective_rag_generate(query):
    """Generate response using selective history retrieval."""

    # 1. Retrieve from knowledge base
    query_emb = retriever.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, kb_embeddings)[0]
    top_indices = torch.topk(scores, k=2).indices
    kb_context = "\n".join([knowledge_base[idx] for idx in top_indices])

    # 2. Get immediate previous exchanges
    immediate = get_immediate_history()
    immediate_text = "\n".join([f"User: {h['query']}\nAssistant: {h['response']}" for h in immediate])

    # 3. Get most relevant past exchanges
    relevant = retrieve_relevant_history(query)
    relevant_text = "\n".join([f"User: {h['query']}\nAssistant: {h['response']}" for h in relevant])

    # 4. Construct prompt
    system_prompt = """You are a technical expert assistant. You must ONLY use the provided context to answer questions. If the context does not contain the answer, respond with 'I don't know'."""

    full_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

Knowledge Base Context:
{kb_context}

Recent Conversation:
{immediate_text}

Relevant Past Exchanges:
{relevant_text}

New Question: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    # 5. Generate
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=1536).to(device)

    with torch.no_grad():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    response = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # 6. Save to history
    save_to_history(query, response)

    return response

# --- Interactive Loop ---

print("\nSelective Retrieval Conversational RAG System Ready!")
print("History is saved and retrieved selectively based on relevance.")
print("Type 'exit' to quit.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() == 'exit':
        print("Goodbye!")
        break

    ai_response = selective_rag_generate(user_input)
    print(f"AI: {ai_response}\n")

Loading models...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Selective Retrieval Conversational RAG System Ready!
History is saved and retrieved selectively based on relevance.
Type 'exit' to quit.

You: what is RAG
AI: RAG stands for Retrieval-Augmented Generation.

You: what is the capital of france?
AI: The capital of France is Paris.

You: what is deep learning?
AI: I don't know.

You: exit
Goodbye!


In [4]:
# ============================================================================
# SECTION 4: EXERCISE 3 (EXTRA) - MULTI-TURN PDF KNOWLEDGE BASE
# ============================================================================

from pypdf import PdfReader
import os
import json
from datetime import datetime

# --- PDF Processing Functions ---

def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file."""
    reader = PdfReader(pdf_path)
    text = ""
    for page_num, page in enumerate(reader.pages):
        text += f"\n[Page {page_num + 1}]\n" + page.extract_text()
    return text

def chunk_text(text, chunk_size=200, overlap=50):
    """Split text into chunks with overlap."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

def process_pdf_folder(folder_path, chunk_size=200):
    """Process all PDFs in a folder and create knowledge base."""
    pdf_kb = []
    pdf_metadata = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            text = extract_text_from_pdf(pdf_path)
            chunks = chunk_text(text, chunk_size=chunk_size)

            for i, chunk in enumerate(chunks):
                pdf_kb.append(chunk)
                pdf_metadata.append({
                    "filename": filename,
                    "chunk_id": i,
                    "source": f"{filename} (Chunk {i})"
                })

    return pdf_kb, pdf_metadata

# --- Multi-Turn PDF RAG ---

PDF_HISTORY_FILE = "pdf_conversation_history.json"

def pdf_rag_with_memory(query, pdf_kb, pdf_embeddings, pdf_metadata):
    """Generate response from PDF knowledge base with conversation memory."""

    # 1. Retrieve from PDF knowledge base
    query_emb = retriever.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, pdf_embeddings)[0]
    top_indices = torch.topk(scores, k=3).indices

    context_chunks = []
    sources = []
    for idx in top_indices:
        context_chunks.append(pdf_kb[idx])
        sources.append(pdf_metadata[idx]["source"])

    context = "\n\n".join(context_chunks)

    # 2. Load conversation history
    try:
        with open(PDF_HISTORY_FILE, 'r') as f:
            history = json.load(f)
    except FileNotFoundError:
        history = []

    # 3. Format recent history
    recent_history = history[-3:]
    history_text = "\n".join([f"User: {h['query']}\nAssistant: {h['response']}" for h in recent_history])

    # 4. Construct prompt
    system_prompt = """You are a helpful assistant that answers questions based on PDF documents. You must ONLY use the provided context from the documents. If the answer is not in the context, say 'I don't know'."""

    full_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

Document Context:
{context}

Recent Conversation:
{history_text}

New Question: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    # 5. Generate
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)

    with torch.no_grad():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    response = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # 6. Save to history
    history.append({
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "response": response,
        "sources": sources
    })

    with open(PDF_HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)

    return response, sources



In [6]:

pdf_folder = "./pdfs"  # Replace with your PDF folder path
pdf_kb, pdf_metadata = process_pdf_folder(pdf_folder, chunk_size=200)
pdf_embeddings = retriever.encode(pdf_kb, convert_to_tensor=True)

print("\nMulti-Turn PDF RAG System Ready!")
print("Type 'exit' to quit.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() == 'exit':
        print("Goodbye!")
        break

    response, sources = pdf_rag_with_memory(user_input, pdf_kb, pdf_embeddings, pdf_metadata)
    print(f"AI: {response}")
    print(f"Sources: {', '.join(sources)}\n")



Multi-Turn PDF RAG System Ready!
Type 'exit' to quit.

You: what is yakada
AI: Yakada is a board games studio dedicated to reviving the intellectual, cultural, and spiritual richness of the global Muslim community through play.
Sources: YAKADA.pdf (Chunk 0), YAKADA.pdf (Chunk 2), YAKADA.pdf (Chunk 1)

You: what is zaman
AI: According to the document, ZAMAN is the first game in the Yakada series, specifically a 56-card timeline-based game where players learn about major events, figures, and eras of Islamic history by placing cards in chronological order.
Sources: YAKADA.pdf (Chunk 2), YAKADA.pdf (Chunk 1), YAKADA.pdf (Chunk 4)

You: exit
Goodbye!
